In [ ]:
# Install system tools
!apt-get update -y -q
!apt-get install -y -q mafft

# Install trimAl
!wget https://github.com/inb/trimAl/releases/download/v1.4.1/trimal.v1.4.1.tar.gz
!tar -xzf trimal.v1.4.1.tar.gz
!mv trimal.v1.4.1/bin/trimal /usr/local/bin/trimal
!chmod +x /usr/local/bin/trimal
!rm trimal.v1.4.1.tar.gz # Clean up
!rm -rf trimal.v1.4.1   # Clean up

# Install IQ-TREE2
!wget https://github.com/iqtree/iqtree2/releases/download/v2.2.2.8/iqtree-2.2.2.8-Linux.tar.gz
!tar -xzf iqtree-2.2.2.8-Linux.tar.gz
!mv iqtree-2.2.2.8-Linux/bin/iqtree2 /usr/local/bin/iqtree2
!chmod +x /usr/local/bin/iqtree2

# Install Python packages
!pip install -q biopython ete3 matplotlib seaborn pandas numpy

# Verify installations
!mafft --version
!trimal --version
!iqtree2 --version | head -2
import Bio
print(f"Biopython {Bio.__version__} installed OK")
print("All tools installed successfully!")

In [ ]:
import os, time, subprocess, re, zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from IPython.display import Image, display
from Bio import Entrez, SeqIO, AlignIO
import Bio.Phylo as Phylo

# ⚠️ CHANGE THIS to your own email address (required by NCBI)
Entrez.email = "eishabibi333@gmail.com"

# Create working directories
WORKDIR = Path("kinesin_phylo")
for sub in ["", "sequences", "alignment", "trees", "figures"]:
    (WORKDIR / sub).mkdir(exist_ok=True)

print("Working directories created:")
for d in sorted(WORKDIR.iterdir()):
    print(f"  {d}")

In [ ]:
# Curated kinesin protein accessions from NCBI
# Sources: Lawrence et al. 2004; Hirokawa et al. 2009
KINESIN_ACCESSIONS = {
    # Kinesin-1 (Conventional kinesin — plus-end directed, axonal transport)
    "KIF5A_Hs":  "NP_004975.2",    # Human KIF5A
    "KIF5B_Hs":  "NP_004512.2",    # Human KIF5B
    "KIF5C_Hs":  "NP_004515.1",    # Human KIF5C
    "KIF5B_Mm":  "NP_032468.1",    # Mouse KIF5B
    "KHC_Dm":    "NP_524456.1",    # Drosophila kinesin heavy chain
    "UNC116_Ce": "NP_001023998.1", # C. elegans unc-116
    # Kinesin-2 (intraflagellar transport, cilia assembly)
    "KIF3A_Hs":  "NP_002254.2",    # Human KIF3A
    "KIF3B_Hs":  "NP_004519.1",    # Human KIF3B
    # Kinesin-3 (vesicle transport in neurons)
    "KIF1A_Hs":  "NP_006547.2",    # Human KIF1A
    "KIF1B_Hs":  "NP_015414.1",    # Human KIF1B
    "UNC104_Ce": "NP_001021409.1", # C. elegans unc-104
    # Kinesin-4
    "KIF4A_Hs":  "NP_001230748.1", # Human KIF4A
    "KIF4B_Hs":  "NP_055735.1",    # Human KIF4B
    # Kinesin-5 (Eg5/BimC family — bipolar spindle assembly)
    "KIF11_Hs":  "NP_004514.2",    # Human KIF11 (Eg5)
    "KLP61F_Dm": "NP_476723.1",    # Drosophila KLP61F
    "CIN8_Sc":   "NP_009400.1",    # Yeast Cin8
    "KIP1_Sc":   "NP_009401.1",    # Yeast Kip1
    # Kinesin-6 (cytokinesis, central spindle)
    "KIF20A_Hs": "NP_005865.1",    # Human KIF20A (MKLP2)
    "KIF23_Hs":  "NP_005379.2",    # Human KIF23 (MKLP1)
    # Kinesin-7 (kinetochore attachment)
    "CENPE_Hs":  "NP_001804.2",    # Human CENP-E
    # Kinesin-8 (microtubule length control)
    "KIF18A_Hs": "NP_690860.1",    # Human KIF18A
    "KIF18B_Hs": "NP_976313.1",    # Human KIF18B
    # Kinesin-13 (microtubule depolymerizing — M-type, internal motor domain)
    "KIF2A_Hs":  "NP_001139526.1", # Human KIF2A
    "KIF2B_Hs":  "NP_055733.1",    # Human KIF2B
    "KIF2C_Hs":  "NP_006836.2",    # Human KIF2C (MCAK)
    "KLP10A_Dm": "NP_524548.1",    # Drosophila KLP10A
    # Kinesin-14 (minus-end directed — C-terminal motor domain)
    "KIFC1_Hs":  "NP_006551.1",    # Human KIFC1 (HSET)
    "KIFC3_Hs":  "NP_005543.1",    # Human KIFC3
    "NCD_Dm":    "NP_524399.1",    # Drosophila Ncd
    "KAR3_Sc":   "NP_009523.1",    # Yeast Kar3
}

# Group labels by family
FAMILIES = {
    "Kinesin-1":  [k for k in KINESIN_ACCESSIONS if k.startswith(("KIF5","KHC","UNC116"))],
    "Kinesin-2":  [k for k in KINESIN_ACCESSIONS if k.startswith("KIF3")],
    "Kinesin-3":  [k for k in KINESIN_ACCESSIONS if k.startswith(("KIF1A","KIF1B","UNC104"))],
    "Kinesin-4":  [k for k in KINESIN_ACCESSIONS if k.startswith("KIF4")],
    "Kinesin-5":  [k for k in KINESIN_ACCESSIONS if k.startswith(("KIF11","KLP61","CIN8","KIP1"))],
    "Kinesin-6":  [k for k in KINESIN_ACCESSIONS if k.startswith(("KIF20","KIF23"))],
    "Kinesin-7":  [k for k in KINESIN_ACCESSIONS if k.startswith("CENPE")],
    "Kinesin-8":  [k for k in KINESIN_ACCESSIONS if k.startswith("KIF18")],
    "Kinesin-13": [k for k in KINESIN_ACCESSIONS if k.startswith(("KIF2","KLP10"))],
    "Kinesin-14": [k for k in KINESIN_ACCESSIONS if k.startswith(("KIFC","NCD","KAR3"))],
}

# Color for each family (used in all plots)
FAMILY_COLORS = {
    "Kinesin-1":  "#E63946", "Kinesin-2":  "#F4A261",
    "Kinesin-3":  "#2A9D8F", "Kinesin-4":  "#457B9D",
    "Kinesin-5":  "#8338EC", "Kinesin-6":  "#FB5607",
    "Kinesin-7":  "#3A86FF", "Kinesin-8":  "#06D6A0",
    "Kinesin-13": "#FFBE0B", "Kinesin-14": "#D62828",
}

# Lookup: label -> family name
LABEL_TO_FAMILY = {m: fam for fam, members in FAMILIES.items() for m in members}

print(f"Total sequences defined: {len(KINESIN_ACCESSIONS)}")
for fam, members in FAMILIES.items():
    print(f"  {fam}: {len(members)} sequences")


In [ ]:
def fetch_protein(accession, label, out_dir, retries=3):
    """Fetch a single protein sequence from NCBI and save as FASTA."""
    for attempt in range(retries):
        try:
            handle = Entrez.efetch(db="protein", id=accession,
                                   rettype="fasta", retmode="text")
            rec = SeqIO.read(handle, "fasta")
            handle.close()
            rec.id = label
            rec.description = f"{label} | {accession}"
            SeqIO.write(rec, out_dir / f"{label}.fasta", "fasta")
            return rec
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2)
    return None

seq_dir = WORKDIR / "sequences"
records, failed = [], []

print("Fetching sequences from NCBI...\n")
for label, acc in KINESIN_ACCESSIONS.items():
    print(f"  {label} ({acc}) ...", end=" ", flush=True)
    rec = fetch_protein(acc, label, seq_dir)
    if rec:
        records.append(rec)
        print(f"OK ({len(rec.seq)} aa)")
    else:
        failed.append(label)
        print("FAILED")
    time.sleep(0.4)  # Polite delay to avoid overloading NCBI servers

# Write all sequences into one combined FASTA file
combined_fasta = WORKDIR / "alignment" / "kinesin_all.fasta"
SeqIO.write(records, combined_fasta, "fasta")

print(f"\nFetch complete: {len(records)} succeeded, {len(failed)} failed")
if failed:
    print(f"Failed accessions: {failed}")

# Plot sequence lengths colored by family
lengths = [(r.id, len(r.seq)) for r in records]
df_len  = pd.DataFrame(lengths, columns=["Label", "Length_aa"])
df_len["Family"] = df_len["Label"].map(LABEL_TO_FAMILY)

fig, ax = plt.subplots(figsize=(12, 6))
colors  = [FAMILY_COLORS.get(f, "#999") for f in df_len["Family"]]
ax.barh(df_len["Label"], df_len["Length_aa"], color=colors)
ax.axvline(df_len["Length_aa"].mean(), color="red", linestyle="--",
           label=f"Mean = {df_len['Length_aa'].mean():.0f} aa")
ax.set_xlabel("Sequence length (amino acids)", fontsize=11)
ax.set_title("Kinesin Protein Sequence Lengths", fontsize=13, fontweight="bold")
patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=patches, loc="lower right", fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(WORKDIR / "figures" / "sequence_lengths.png", dpi=150)
plt.show()

In [ ]:
# MAFFT --auto selects the best algorithm based on input size and structure
# For conserved motor domains it typically uses L-INS-i
aligned_fasta = WORKDIR / "alignment" / "kinesin_aligned.fasta"

cmd = ["mafft", "--auto", "--reorder", "--thread", "-1", str(combined_fasta)]
print("Running MAFFT alignment...")
print(" ".join(cmd))

with open(aligned_fasta, "w") as out_f:
    result = subprocess.run(cmd, stdout=out_f, stderr=subprocess.PIPE, text=True)

if result.returncode == 0:
    aln = AlignIO.read(aligned_fasta, "fasta")
    print(f"\nAlignment complete!")
    print(f"  Number of sequences : {len(aln)}")
    print(f"  Alignment length    : {aln.get_alignment_length()} columns")
else:
    print("MAFFT error:", result.stderr[-300:])

In [ ]:
# No trimAl needed — pure Python gap-based trimming
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

trimmed_fasta = WORKDIR / "alignment" / "kinesin_trimmed.fasta"

# Convert alignment to numpy array
aln_arr_raw = np.array([list(str(r.seq)) for r in aln])
n_seq_raw, n_col_raw = aln_arr_raw.shape

print(f"Original alignment: {n_seq_raw} sequences x {n_col_raw} columns")

# Keep only columns where gap fraction <= 50%
keep_cols = []
for c in range(n_col_raw):
    col = aln_arr_raw[:, c]
    gap_frac = (col == "-").sum() / n_seq_raw
    if gap_frac <= 0.5:
        keep_cols.append(c)

print(f"Columns kept (<=50% gaps): {len(keep_cols)}")
print(f"Columns removed           : {n_col_raw - len(keep_cols)} ({100*(n_col_raw-len(keep_cols))/n_col_raw:.1f}%)")

# Build trimmed alignment
trimmed_records = []
for rec in aln:
    seq_str = str(rec.seq)
    new_seq = "".join(seq_str[c] for c in keep_cols)
    trimmed_records.append(
        SeqRecord(Seq(new_seq), id=rec.id, description="")
    )

SeqIO.write(trimmed_records, trimmed_fasta, "fasta")
trimmed_aln = AlignIO.read(trimmed_fasta, "fasta")
print(f"\nTrimmed alignment saved: {trimmed_fasta}")

# Calculate conservation and gap fraction per column
aln_arr = np.array([list(str(r.seq)) for r in trimmed_aln])
n_seq, n_col = aln_arr.shape

conservation = []
gap_fraction = []
for c in range(n_col):
    col   = aln_arr[:, c]
    nogap = col[col != "-"]
    if len(nogap):
        vals, counts = np.unique(nogap, return_counts=True)
        conservation.append(counts.max() / n_seq)
    else:
        conservation.append(0)
    gap_fraction.append((col == "-").sum() / n_seq)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax1.fill_between(range(n_col), conservation, alpha=0.75, color="steelblue")
ax1.axhline(0.8, color="orange", linestyle="--", lw=0.8, label="80% threshold")
ax1.set_ylabel("Conservation")
ax1.set_ylim(0, 1)
ax1.legend(fontsize=9)
ax1.set_title("Alignment Quality — Kinesin Motor Domain", fontweight="bold", fontsize=13)

ax2.fill_between(range(n_col), gap_fraction, alpha=0.75, color="salmon")
ax2.set_ylabel("Gap fraction")
ax2.set_ylim(0, 1)
ax2.set_xlabel("Alignment column")

plt.tight_layout()
plt.savefig(WORKDIR / "figures" / "conservation.png", dpi=150)
plt.show()

high_cons = sum(c >= 0.8 for c in conservation)
print(f"\nHighly conserved columns (>=80%): {high_cons}/{n_col} ({100*high_cons/n_col:.1f}%)")
print("\nCell 6 complete — proceed to Cell 7 (IQ-TREE2)")

In [ ]:
# IQ-TREE2 settings:
#   -m TEST    : ModelFinder automatically selects the best substitution model
#   -B 1000    : 1000 ultrafast bootstrap replicates (UFBoot)
#   -alrt 1000 : SH-aLRT branch support test
#   --bnni     : further optimize UFBoot trees with NNI moves
#   -T AUTO    : auto-detect and use all available CPU threads

tree_prefix = str(WORKDIR / "trees" / "kinesin")

cmd = [
    "iqtree2",
    "-s",    str(trimmed_fasta),
    "-m",    "TEST",
    "-B",    "1000",
    "-alrt", "1000",
    "--bnni",
    "-T",    "AUTO",
    "--prefix", tree_prefix,
    "-redo"
]

print("Running IQ-TREE2... (this will take 5-10 minutes)\n")
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print("IQ-TREE2 complete!")
    for line in result.stdout.split("\n"):
        if any(k in line for k in ["Best-fit model", "Log-likelihood", "wall-clock"]):
            print(" ", line.strip())
else:
    print("IQ-TREE2 error:")
    print(result.stderr[-1000:])

print("\nOutput files generated:")
for f in sorted((WORKDIR / "trees").iterdir()):
    print(f"  {f.name}")

In [ ]:
treefile = tree_prefix + ".treefile"

# Read and sort tree for visual clarity
tree = Phylo.read(treefile, "newick")
tree.ladderize()

fig, ax = plt.subplots(figsize=(12, 14))
Phylo.draw(tree, axes=ax, do_show=False,
    label_colors=lambda c: FAMILY_COLORS.get(
        LABEL_TO_FAMILY.get(c, ""), "black") if c else "black")

ax.set_title("Maximum-Likelihood Phylogeny — Kinesin Motor Proteins",
             fontsize=13, fontweight="bold", pad=15)

# Family color legend
patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=patches, loc="lower left", fontsize=8,
          title="Kinesin Family", title_fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig(WORKDIR / "figures" / "tree_biopython.png", dpi=150, bbox_inches="tight")
plt.show()
print("Tree saved to figures/tree_biopython.png")

In [ ]:
# Extract bootstrap support values from all internal (non-leaf) nodes
tree_bp = Phylo.read(treefile, "newick")
support_values = [
    float(c.confidence) for c in tree_bp.find_clades()
    if c.confidence is not None and not c.is_terminal()
]
sa = np.array(support_values)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Histogram of all support values
ax1.hist(sa, bins=20, color="steelblue", edgecolor="white")
ax1.axvline(70, color="orange", linestyle="--", label="70% threshold")
ax1.axvline(95, color="red",    linestyle="--", label="95% threshold")
ax1.set_xlabel("Bootstrap support (%)")
ax1.set_ylabel("Number of nodes")
ax1.set_title("Bootstrap Support Distribution", fontweight="bold")
ax1.legend()

# Pie chart — three support categories
if len(sa) > 0:
    low  = (sa < 70).sum()
    med  = ((sa >= 70) & (sa < 95)).sum()
    high = (sa >= 95).sum()
    ax2.pie([low, med, high],
            labels=["<70% (weak)", "70-95% (moderate)", ">=95% (strong)"],
            colors=["#FF6B6B", "#FFD93D", "#6BCB77"],
            autopct="%1.1f%%", startangle=140)
    ax2.set_title("Support Categories", fontweight="bold")
else:
    ax2.set_title("No Bootstrap Support Data", fontweight="bold")
    ax2.text(0.5, 0.5, "No internal nodes with confidence values found.",
             horizontalalignment='center', verticalalignment='center',
             transform=ax2.transAxes, fontsize=10, color='gray')


plt.suptitle("Kinesin Phylogeny — Bootstrap Confidence Assessment", fontsize=13)
plt.tight_layout()
plt.savefig(WORKDIR / "figures" / "bootstrap.png", dpi=150)
plt.show()

# Add a check for printing statistics as well
if len(sa) > 0:
    print(f"Total internal nodes : {len(sa)}")
    print(f"  Weak   (<70%)      : {low}  ({100*low/len(sa):.1f}%)")
    print(f"  Moderate (70-95%)  : {med}  ({100*med/len(sa):.1f}%")
    print(f"  Strong (>=95%)     : {high}  ({100*high/len(sa):.1f}%")
else:
    print("No internal nodes with confidence values found for statistical summary.")

In [ ]:
# Load trimmed alignment and compute pairwise identity matrix
aln_r  = AlignIO.read(trimmed_fasta, "fasta")
labels = [r.id for r in aln_r]
n      = len(aln_r)
arr    = np.array([list(str(r.seq)) for r in aln_r])

# Compute identity for each pair, ignoring gap positions
ident = np.zeros((n, n))
for i in range(n):
    for j in range(i, n):
        mask = (arr[i] != "-") & (arr[j] != "-")
        if mask.sum():
            v = (arr[i][mask] == arr[j][mask]).sum() / mask.sum()
            ident[i, j] = ident[j, i] = v

df_id = pd.DataFrame(ident * 100, index=labels, columns=labels)

fig, ax = plt.subplots(figsize=(15, 13))
sns.heatmap(df_id, ax=ax,
            cmap="RdYlGn", vmin=20, vmax=100,
            linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Sequence Identity (%)", "shrink": 0.7})
ax.set_title("Pairwise Sequence Identity — Kinesin Motor Proteins",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(WORKDIR / "figures" / "identity_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Three universally conserved kinesin motor domain sequence motifs:
#   P-loop    : GxxxxGKT/S — binds the phosphate groups of ATP
#   Switch I  : NxxSSR     — senses the gamma-phosphate of ATP
#   Switch II : DxxG       — couples ATP hydrolysis to conformational change
MOTIFS = {
    "P-loop (GxxxxGKT/S)": r"G.{4}GK[TS]",
    "Switch I (NxxSSR)":   r"N.{2}SS[RK]",
    "Switch II (DxxG)":    r"D.{2}G",
}

rows = []
for rec in records:
    row = {
        "Protein": rec.id,
        "Family":  LABEL_TO_FAMILY.get(rec.id, "Unknown"),
        "Length":  len(rec.seq)
    }
    for name, pat in MOTIFS.items():
        hits = re.findall(pat, str(rec.seq))
        row[name] = ", ".join(hits) if hits else "NOT FOUND"
    rows.append(row)

df_motif = pd.DataFrame(rows)

# Style missing motifs in red for easy spotting
def style_missing(v):
    return "background-color:#FFCCCC; color:red" if v == "NOT FOUND" else ""

display(df_motif.style.applymap(
    style_missing,
    subset=[c for c in df_motif.columns if c not in ["Protein", "Family", "Length"]]
).set_caption("Conserved Kinesin Motor Domain Motifs"))

print()
for m in MOTIFS:
    found = (df_motif[m] != "NOT FOUND").sum()
    print(f"{m:30s}: found in {found}/{len(df_motif)} sequences")

In [ ]:
zip_path = "kinesin_phylo_results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    # Include alignment, tree, and figure files
    for sub in ["alignment", "trees", "figures"]:
        for f in (WORKDIR / sub).iterdir():
            zf.write(f, f"{sub}/{f.name}")
    # Include motif table as CSV
    df_motif.to_csv("kinesin_motifs.csv", index=False)
    zf.write("kinesin_motifs.csv")

print(f"ZIP archive ready: {zip_path}")

# Trigger download inside Google Colab
from google.colab import files
files.download(zip_path)
print("Download started!")